In [3]:
# Paso 1:  archivo LAS para convertirlo a un dataframe
# Aquí cargo el archivo LAS que contiene los datos del pozo
import lasio
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Especifico la ruta del archivo LAS que quiero analizar.
las_file_path = "data/well_logs/WA1.LAS"
las = lasio.read(las_file_path)

# Convierto los datos del archivo LAS a un DataFrame de pandas 
df = las.df().dropna()

# Mostrar las primeras filas del DataFrame
# Verifico que los datos se hayan cargado correctamente.
df.head()

ModuleNotFoundError: No module named 'lasio'

In [ ]:
# Paso 2
# Aquí calculo estadísticas descriptivas para entender mejor los datos del pozo
# Esto me permite identificar valores como el promedio, mínimo, máximo y otros
statistics = df.describe()

# Reviso las estadísticas para identificar tendencias o valores interesantes
statistics

In [ ]:
# Paso 3:

# Aquí calculo la concentración volumétrica de lutita usando el registro Gamma Ray

# Esto es importante para identificar las zonas con mayor contenido de lutita

# Definir los valores mínimos y máximos del registro gamma ray
# Calculo los valores mínimo y máximo del registro GR para usarlos en la fórmula.
GR_min = df['GR'].min()

GR_max = df['GR'].max()

# Aplico la fórmula para calcular V_sh y lo añado como una nueva columna en el DataFrame.
df['V_sh'] = (df['GR'] - GR_min) / (GR_max - GR_min)

# Verifico que la columna V_sh se haya calculado correctamente.
df[['GR', 'V_sh']].head()

In [ ]:
# Paso 4
# Aquí identifico las propiedades de la lutita pura y calculo las porosidades corregidas.
# Encuentro los valores máximos de GR y sus correspondientes valores de RHOB y NPHI.
GR_sh = df['GR'].max()
RHOB_sh = df.loc[df['GR'] == GR_sh, 'RHOB'].values[0]
NPHI_sh = df.loc[df['GR'] == GR_sh, 'NPHI'].values[0]

# Uso las fórmulas dadas para calcular las porosidades corregidas.
df['phi_D'] = (df['RHOB'] - 1.65) / (2.65 - 1.65)
df['phi_D_c'] = (df['phi_D'] - df['V_sh'] * (RHOB_sh - 1.65) / (2.65 - 1.65)) / (1 - df['V_sh'])
df['phi_N_c'] = (df['NPHI'] - df['V_sh'] * NPHI_sh) / (1 - df['V_sh'])
df['phi'] = ((df['phi_D_c']**2 + df['phi_N_c']**2) / 2)**0.5

# Verifico que las porosidades corregidas se hayan calculado correctamente.
df[['GR', 'RHOB', 'NPHI', 'V_sh', 'phi_D', 'phi_D_c', 'phi_N_c', 'phi']].head()

In [ ]:
# Paso  5
# aquí creo los track para ver los datos del pozo
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(20, 10), sharey=True)

# Track 1: Gamma Ray y V_sh
# Visualizo el registro Gamma Ray junto con la concentración volumétrica de lutita.
axes[0].plot(df['GR'], df.index, label='Gamma Ray (GR)', color='green')
axes[0].plot(df['V_sh'], df.index, label='V_sh', color='blue')
axes[0].set_xlabel('GR / V_sh')
axes[0].set_ylabel('Depth')
axes[0].invert_yaxis()
axes[0].legend()
axes[0].grid()

# Track 2: RHOB y NPHI
# Comparo la densidad y la porosidad neutrónica para identificar tendencias.
axes[1].plot(df['RHOB'], df.index, label='RHOB', color='red')
axes[1].plot(df['NPHI'], df.index, label='NPHI', color='purple')
axes[1].set_xlabel('RHOB / NPHI')
axes[1].invert_yaxis()
axes[1].legend()
axes[1].grid()

# Track 3: Resistividad (LL8, ILM, ILD)
# Uso una escala logarítmica para visualizar los registros de resistividad.
axes[2].semilogx(df['LL8'], df.index, label='LL8 (Shallow)', color='orange')
axes[2].semilogx(df['ILM'], df.index, label='ILM (Medium)', color='brown')
axes[2].semilogx(df['ILD'], df.index, label='ILD (Deep)', color='black')
axes[2].set_xlabel('Resistivity (log scale)')
axes[2].invert_yaxis()
axes[2].legend()
axes[2].grid()

# Track 4: Otros registros interesantes
# Aquí incluyo un registro adicional para análisis complementario.
axes[3].plot(df['CALI'], df.index, label='CALI (Caliper)', color='cyan')
axes[3].set_xlabel('CALI')
axes[3].invert_yaxis()
axes[3].legend()
axes[3].grid()

# Uso tight_layout para asegurarme de que los gráficos no se solapen.
plt.tight_layout()
plt.show()

# Actividad 3: Modelado espacial del reservorio
En esta actividad, se realiza un análisis espacial del reservorio utilizando datos de permeabilidad y máscaras de celdas activas.

In [ ]:
# Importo las librerías necesarias para la manipulación de datos y la visualización.
import numpy as np
import matplotlib.pyplot as plt
import os

# Defino las rutas relativas para los archivos de datos
base_dir = "../../../Course Content/Week 1 - Introduction/data/egg_model/2D"
perm_file = os.path.join(base_dir, "PERM29.npy")
mask_file = os.path.join(base_dir, "ACTIVE.npy")

# Cargo la realización de permeabilidad 2D y la máscara de celdas activas.
permeabilidad = np.load(perm_file)
mascara_activa = np.load(mask_file)

# Muestro las formas de los arreglos cargados para verificar que se cargaron correctamente.
print("Forma de la realización de permeabilidad:", permeabilidad.shape)
print("Forma de la máscara activa:", mascara_activa.shape)

In [ ]:
# Creo un subplot 1x3 para visualizar los datos
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Visualizo la realización de permeabilidad.
im1 = axes[0].imshow(permeabilidad, cmap='viridis')
axes[0].set_title("Realización de Permeabilidad")
fig.colorbar(im1, ax=axes[0])

# Visualizo la máscara de celdas activas.
im2 = axes[1].imshow(mascara_activa, cmap='binary')
axes[1].set_title("Máscara de Celdas Activas")
fig.colorbar(im2, ax=axes[1])

# Visualizo la permeabilidad enmascarada.
permeabilidad_enmascarada = np.ma.masked_where(mascara_activa == 0, permeabilidad)
im3 = axes[2].imshow(permeabilidad_enmascarada, cmap='plasma')
axes[2].set_title("Permeabilidad Enmascarada")
fig.colorbar(im3, ax=axes[2])

# Ajusto el diseño para evitar solapamientos y muestro el gráfico final.
plt.tight_layout()
plt.show()